In [ ]:
import re
import ast
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
import nltk
for pkg in ["punkt", "stopwords", "wordnet", "omw-1.4", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

In [ ]:
DATA_PATH     = "nyt_articles.csv"   # adjust path if needed
OUTPUT_PREFIX = "nyt_articles"

print("=" * 60)
print("STEP 1 – Loading data")
print("=" * 60)

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Columns: {list(df.columns)}\n")

STEP 1 – Loading data
Rows: 4,263  |  Columns: 19
Columns: ['article_id', 'pub_date', 'section', 'subsection', 'type', 'document_type', 'headline', 'abstract', 'snippet', 'lead_paragraph', 'byline', 'source', 'news_desk', 'word_count', 'keywords', 'web_url', 'thumbnail_url', 'print_page', 'print_section']



In [ ]:
# ── Filter 1: Keep only rows where document_type is 'Article' ─────────────────
# NOTE: this dataset has both 'type' and 'document_type' columns.
# 'document_type' is more reliable here — it marks true articles vs corrections,
# quotes of the day, spelling bee posts, word-of-the-day entries, etc.
if "document_type" in df.columns:
    before = len(df)
    df = df[df["document_type"].str.lower() == "article"].copy()
    print(f"After filtering to document_type='Article': {len(df):,} rows "
          f"(removed {before - len(df):,})\n")

# ── Filter 2: Drop low-substance news desks ───────────────────────────────────
# These desks produce filler content (puzzles, corrections, word-of-the-day)
# that would pollute topic modeling with meaningless words
EXCLUDE_DESKS = {
    "games", "corrections", "learning", "summary", "obits",
    "obituaries", "crosswords"
}
if "news_desk" in df.columns:
    before = len(df)
    df = df[~df["news_desk"].str.lower().isin(EXCLUDE_DESKS)].copy()
    print(f"After removing low-substance news desks: {len(df):,} rows "
          f"(removed {before - len(df):,})\n")

# ── Filter 3: Drop rows with very low word counts (likely stubs/live blog posts)
if "word_count" in df.columns:
    before = len(df)
    df = df[pd.to_numeric(df["word_count"], errors="coerce").fillna(0) >= 100].copy()
    print(f"After removing short stubs (word_count < 100): {len(df):,} rows "
          f"(removed {before - len(df):,})\n")

# ── Drop rows with no headline AND no abstract ────────────────────────────────
df = df.dropna(subset=["headline", "abstract"], how="all").copy()
print(f"After dropping empty headline+abstract rows: {len(df):,} rows\n")

# ── Show section breakdown ────────────────────────────────────────────────────
if "section" in df.columns:
    print("Articles per section:")
    print(df["section"].value_counts().to_string())
    print()

After filtering to document_type='Article': 4,099 rows (removed 164)

After removing low-substance news desks: 3,658 rows (removed 441)

After removing short stubs (word_count < 100): 3,635 rows (removed 23)

After dropping empty headline+abstract rows: 3,635 rows

Articles per section:
section
U.S.              739
World             628
Opinion           309
New York          276
Business          275
Arts              253
Food              122
Style             117
Movies            108
Podcasts           90
Books              78
Real Estate        74
Climate            74
Well               69
Briefing           66
Technology         43
Science            42
Magazine           40
Travel             39
Theater            38
Health             36
T Magazine         35
Weather            28
Special Series     16
Times Insider      16
The Upshot          6
Fashion             5
Your Money          3
En español          3
Sports              2
Admin               2
Obituaries          2


In [ ]:
# 2. SELECT & COMBINE TEXT FIELDS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 2 – Selecting and combining text fields")
print("=" * 60)

def parse_keywords(val):
    """
    In this dataset, keywords is a plain comma-separated string
    (e.g. 'Television; Trump, Donald J; Ukraine').
    We split on semicolons and return a clean space-joined string.
    NOTE: Unlike mostpopular/topstories, this is NOT a Python list —
    so we use string splitting instead of ast.literal_eval.
    """
    if pd.isna(val) or str(val).strip() == "":
        return ""
    # Split on semicolons, strip whitespace, rejoin with spaces
    parts = [p.strip() for p in str(val).split(";") if p.strip()]
    return " ".join(parts)

df["keywords_text"] = df["keywords"].apply(parse_keywords)

# This dataset has more text fields than the others — we use all of them:
#   headline    → equivalent to 'title' in other datasets
#   abstract    → brief summary
#   snippet     → short excerpt (often same as abstract, adds a little extra)
#   lead_paragraph → first paragraph, often the most informative sentence
#   keywords    → topic tags assigned by NYT editors
df["combined_text"] = (
    df["headline"].fillna("") + " " +
    df["abstract"].fillna("") + " " +
    df["snippet"].fillna("") + " " +
    df["lead_paragraph"].fillna("") + " " +
    df["keywords_text"]
)

print(f"Sample combined text:\n{df['combined_text'].iloc[0][:400]}\n")

STEP 2 – Selecting and combining text fields
Sample combined text:
‘It: Welcome to Derry’ Season 1, Episode 6 Recap: Daddy’s Little Girl Ingrid’s connection to the evil entity known as It is revealed, among other dangerous secrets.   Television It: Welcome to Derry (TV Program)



In [ ]:
# 3. TEXT PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 3 – Text preprocessing")
print("=" * 60)

STOP_WORDS = set(stopwords.words("english"))

# Domain-specific stopwords
CUSTOM_STOPS = {
    "new", "york", "times", "nyt", "said", "say", "mr", "ms", "dr",
    "like", "one", "also", "year", "years", "could", "would", "get",
    "make", "many", "much", "way", "time", "day", "week", "month",
    # metadata noise seen in keywords field for this dataset
    "internal", "open", "access", "storyline", "live", "detached",
    "neutral", "audio",
}
STOP_WORDS.update(CUSTOM_STOPS)

lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    """
    Full preprocessing pipeline:
      - Lowercase
      - Remove URLs, HTML tags, punctuation, digits
      - Tokenize
      - Remove stopwords
      - Lemmatize
      - Keep only tokens with 3+ characters
    Returns a single cleaned string.
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)       # remove URLs
    text = re.sub(r"<[^>]+>", "", text)              # remove HTML tags
    text = re.sub(r"[^a-z\s]", " ", text)            # keep letters only
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(tok)
        for tok in tokens
        if tok not in STOP_WORDS and len(tok) >= 3
    ]
    return " ".join(tokens)


df["clean_text"] = df["combined_text"].apply(preprocess_text)

# Drop rows where cleaning produced an empty string
df = df[df["clean_text"].str.strip() != ""].copy()
print(f"Documents after cleaning: {len(df):,}")
print(f"\nSample cleaned text:\n{df['clean_text'].iloc[0][:300]}\n")

STEP 3 – Text preprocessing
Documents after cleaning: 3,635

Sample cleaned text:
welcome derry season episode recap daddy little girl ingrid connection evil entity known revealed among dangerous secret television welcome derry program



In [ ]:
# 4. VECTORIZATION
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 4 – Vectorization (Bag-of-Words for LDA, TF-IDF for NMF)")
print("=" * 60)

# ── Count Vectorizer for LDA ──────────────────────────────────────────────────
count_vec = CountVectorizer(
    max_df=0.90,        # ignore terms appearing in >90% of docs
    min_df=5,           # ignore terms in fewer than 5 docs
    max_features=8000,
    ngram_range=(1, 2)  # unigrams + bigrams
)
doc_term_matrix_count = count_vec.fit_transform(df["clean_text"])
count_vocab = count_vec.get_feature_names_out()
print(f"Count vocabulary size: {len(count_vocab):,}")

# ── TF-IDF Vectorizer for NMF ─────────────────────────────────────────────────
tfidf_vec = TfidfVectorizer(
    max_df=0.90,
    min_df=5,
    max_features=8000,
    ngram_range=(1, 2)
)
doc_term_matrix_tfidf = tfidf_vec.fit_transform(df["clean_text"])
tfidf_vocab = tfidf_vec.get_feature_names_out()
print(f"TF-IDF vocabulary size: {len(tfidf_vocab):,}\n")

STEP 4 – Vectorization (Bag-of-Words for LDA, TF-IDF for NMF)
Count vocabulary size: 5,640
TF-IDF vocabulary size: 5,640



In [ ]:
# 5. TOPIC MODELING
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 5 – Topic Modeling (LDA + NMF)")
print("=" * 60)

N_TOPICS     = 10     # larger corpus supports more granular topics
N_TOP_WORDS  = 15
RANDOM_STATE = 42


def display_topics(model, feature_names, n_top_words, model_name="Model"):
    """Print top words for each topic."""
    print(f"\n{'─'*50}")
    print(f"  {model_name} – Top {n_top_words} words per topic")
    print(f"{'─'*50}")
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[: -n_top_words - 1 : -1]
        top_words   = [feature_names[i] for i in top_indices]
        print(f"  Topic {topic_idx:2d}: {', '.join(top_words)}")
    print()


# ── LDA ───────────────────────────────────────────────────────────────────────
print("Training LDA …")
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=20,
    learning_method="online",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
lda_topic_matrix = lda.fit_transform(doc_term_matrix_count)
display_topics(lda, count_vocab, N_TOP_WORDS, model_name="LDA")

df["lda_dominant_topic"]    = lda_topic_matrix.argmax(axis=1)
df["lda_topic_probability"] = lda_topic_matrix.max(axis=1)

# ── NMF ───────────────────────────────────────────────────────────────────────
print("Training NMF …")
nmf = NMF(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    max_iter=500
)
nmf_topic_matrix = nmf.fit_transform(doc_term_matrix_tfidf)
display_topics(nmf, tfidf_vocab, N_TOP_WORDS, model_name="NMF")

df["nmf_dominant_topic"]    = nmf_topic_matrix.argmax(axis=1)
df["nmf_topic_probability"] = nmf_topic_matrix.max(axis=1)

STEP 5 – Topic Modeling (LDA + NMF)
Training LDA …

──────────────────────────────────────────────────
  LDA – Top 15 words per topic
──────────────────────────────────────────────────
  Topic  0: inc, university, intelligence, artificial, artificial intelligence, college, computer, technology, woman, internet, company, college university, airline, student, computer internet
  Topic  1: book, australia, israel, health, child, sydney, gaza, rate, sydney australia, price, literature, book literature, fee, fare, hanukkah
  Topic  2: murder, shooting, brown, care, police, attempted, university, syria, attack, city, homicide, travel, attempted murder, brown university, murder attempted
  Topic  3: thousand, two, twenty, two thousand, thousand twenty, research, five, twenty five, global, energy, warming, global warming, france, gas, britain
  Topic  4: party, election, mamdani, republican, city, democratic, mayor, zohran, house, government, type, politics, republican party, content, content 

In [ ]:
# 6. TOPIC LABELS
# ─────────────────────────────────────────────────────────────────────────────
# Update these after reviewing the topic keywords printed above
LDA_TOPIC_LABELS = {i: f"LDA_Topic_{i}" for i in range(N_TOPICS)}
NMF_TOPIC_LABELS = {i: f"NMF_Topic_{i}" for i in range(N_TOPICS)}

df["lda_topic_label"] = df["lda_dominant_topic"].map(LDA_TOPIC_LABELS)
df["nmf_topic_label"] = df["nmf_dominant_topic"].map(NMF_TOPIC_LABELS)

In [ ]:
# 7. PLOTS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 6 – Generating plots")
print("=" * 60)

# ── Topic distribution bar chart ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, title in [
    (axes[0], "lda_dominant_topic", f"LDA Topic Distribution (k={N_TOPICS})"),
    (axes[1], "nmf_dominant_topic", f"NMF Topic Distribution (k={N_TOPICS})"),
]:
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color="steelblue", edgecolor="white")
    ax.set_xlabel("Topic ID")
    ax.set_ylabel("Article Count")
    ax.set_title(title)
    ax.set_xticks(range(N_TOPICS))

plt.suptitle("NYT Articles – Topic Distribution", fontsize=13, y=1.02)
plt.tight_layout()
dist_plot_path = f"{OUTPUT_PREFIX}_topic_distribution.png"
plt.savefig(dist_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {dist_plot_path}")
plt.close()

# ── LDA keyword bars per topic ────────────────────────────────────────────────
n_cols = 5
n_rows = (N_TOPICS + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, n_rows * 4))
axes = axes.flatten()
for topic_idx, topic in enumerate(lda.components_):
    top_n       = 10
    top_indices = topic.argsort()[: -top_n - 1 : -1]
    top_words   = [count_vocab[i] for i in top_indices]
    top_vals    = [topic[i] for i in top_indices]
    axes[topic_idx].barh(top_words[::-1], top_vals[::-1], color="cornflowerblue")
    axes[topic_idx].set_title(f"LDA Topic {topic_idx}", fontsize=10)
    axes[topic_idx].tick_params(labelsize=8)

for idx in range(N_TOPICS, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("LDA – Top 10 Keywords per Topic", fontsize=13)
plt.tight_layout()
kw_plot_path = f"{OUTPUT_PREFIX}_lda_keywords.png"
plt.savefig(kw_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved: {kw_plot_path}")
plt.close()

# ── Topic breakdown by section ────────────────────────────────────────────────
if "section" in df.columns:
    section_topic = (
        df.groupby(["section", "lda_dominant_topic"])
        .size()
        .unstack(fill_value=0)
    )
    top_sections  = df["section"].value_counts().head(8).index
    section_topic = section_topic.loc[section_topic.index.isin(top_sections)]

    fig, ax = plt.subplots(figsize=(14, 6))
    section_topic.plot(kind="bar", ax=ax, colormap="tab10", width=0.8)
    ax.set_title("LDA Topic Distribution by NYT Section", fontsize=13)
    ax.set_xlabel("Section")
    ax.set_ylabel("Article Count")
    ax.legend(title="Topic ID", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    section_plot_path = f"{OUTPUT_PREFIX}_topics_by_section.png"
    plt.savefig(section_plot_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {section_plot_path}")
    plt.close()

# ── Topic trend over time (monthly) ──────────────────────────────────────────
# This dataset spans Dec 2025 – Jan 2026, so a monthly trend is meaningful
if "pub_date" in df.columns:
    df["pub_month"] = pd.to_datetime(
        df["pub_date"], errors="coerce"
    ).dt.to_period("M").astype(str)

    time_topic = (
        df.groupby(["pub_month", "lda_dominant_topic"])
        .size()
        .unstack(fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    time_topic.plot(kind="bar", ax=ax, colormap="tab10", width=0.8)
    ax.set_title("LDA Topic Distribution by Month", fontsize=13)
    ax.set_xlabel("Month")
    ax.set_ylabel("Article Count")
    ax.legend(title="Topic ID", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.xticks(rotation=0)
    plt.tight_layout()
    time_plot_path = f"{OUTPUT_PREFIX}_topics_by_month.png"
    plt.savefig(time_plot_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {time_plot_path}")
    plt.close()

STEP 6 – Generating plots
Saved: nyt_articles_topic_distribution.png
Saved: nyt_articles_lda_keywords.png
Saved: nyt_articles_topics_by_section.png


/tmp/ipykernel_5120/3814267352.py:79: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ).dt.to_period("M").astype(str)


Saved: nyt_articles_topics_by_month.png


In [ ]:
# 8. SAVE OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 7 – Saving outputs")
print("=" * 60)

# ── Article-level results ─────────────────────────────────────────────────────
output_cols = [
    "headline", "abstract", "section", "subsection", "pub_date",
    "news_desk", "word_count", "type",
    "clean_text",
    "lda_dominant_topic", "lda_topic_probability", "lda_topic_label",
    "nmf_dominant_topic", "nmf_topic_probability", "nmf_topic_label",
]
output_cols = [c for c in output_cols if c in df.columns]

results_path = f"{OUTPUT_PREFIX}_with_topics.csv"
df[output_cols].to_csv(results_path, index=False)
print(f"Saved article-level results: {results_path}")

# ── Topic-keyword summary ─────────────────────────────────────────────────────
rows = []
for model_name, model, vocab in [("LDA", lda, count_vocab), ("NMF", nmf, tfidf_vocab)]:
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[: -N_TOP_WORDS - 1 : -1]
        top_words   = ", ".join([vocab[i] for i in top_indices])
        rows.append({"model": model_name, "topic_id": topic_idx, "top_keywords": top_words})

topics_df   = pd.DataFrame(rows)
topics_path = f"{OUTPUT_PREFIX}_topic_keywords.csv"
topics_df.to_csv(topics_path, index=False)
print(f"Saved topic-keyword summary: {topics_path}")

# ── Topic distribution summary ────────────────────────────────────────────────
lda_dist         = df["lda_dominant_topic"].value_counts().reset_index()
lda_dist.columns = ["topic_id", "article_count"]
lda_dist["model"] = "LDA"

nmf_dist         = df["nmf_dominant_topic"].value_counts().reset_index()
nmf_dist.columns = ["topic_id", "article_count"]
nmf_dist["model"] = "NMF"

dist_path = f"{OUTPUT_PREFIX}_topic_distribution.csv"
pd.concat([lda_dist, nmf_dist]).to_csv(dist_path, index=False)
print(f"Saved topic distribution: {dist_path}")

# ── Section-topic breakdown ───────────────────────────────────────────────────
if "section" in df.columns:
    section_path = f"{OUTPUT_PREFIX}_section_topic_breakdown.csv"
    df.groupby(["section", "lda_dominant_topic"]).size().reset_index(
        name="article_count"
    ).to_csv(section_path, index=False)
    print(f"Saved section-topic breakdown: {section_path}")

# ── Monthly topic trend ───────────────────────────────────────────────────────
if "pub_month" in df.columns:
    monthly_path = f"{OUTPUT_PREFIX}_monthly_topic_trend.csv"
    df.groupby(["pub_month", "lda_dominant_topic"]).size().reset_index(
        name="article_count"
    ).to_csv(monthly_path, index=False)
    print(f"Saved monthly topic trend: {monthly_path}")

STEP 7 – Saving outputs
Saved article-level results: nyt_articles_with_topics.csv
Saved topic-keyword summary: nyt_articles_topic_keywords.csv
Saved topic distribution: nyt_articles_topic_distribution.csv
Saved section-topic breakdown: nyt_articles_section_topic_breakdown.csv
Saved monthly topic trend: nyt_articles_monthly_topic_trend.csv


In [ ]:
# 9. QUICK SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Sample articles per LDA topic (first 2 per topic):")
print("=" * 60)
for t in range(N_TOPICS):
    subset = df[df["lda_dominant_topic"] == t][
        ["headline", "section", "lda_topic_probability"]
    ].head(2)
    print(f"\n  Topic {t}:")
    for _, row in subset.iterrows():
        section_label = f"[{row['section']}]" if "section" in row else ""
        print(f"    [{row['lda_topic_probability']:.2f}] {section_label} {row['headline']}")

print("\n✓ All done! Outputs written to the working directory.")
print("  Next step: review topic keywords above, assign meaningful labels,")
print("  and update LDA_TOPIC_LABELS / NMF_TOPIC_LABELS in Section 6.\n")


Sample articles per LDA topic (first 2 per topic):

  Topic 0:
    [0.96] [U.S.] Winter Storm Is Expected to Bring Snow and Rain to Much of the Northeast
    [0.46] [Opinion] I’m an A.I. Developer. Here’s How I’m Raising My Son.

  Topic 1:
    [0.35] [Business] Get Ready, America: Here Come China’s Food and Drink Chains
    [0.78] [Well] A Smartphone Before Age 12 Could Carry Health Risks, Study Says

  Topic 2:
    [0.95] [New York] At Mangione’s Hearing, Giddy Fans Whisper and Prison Guards Testify
    [0.39] [New York] Gaming Board Recommends All 3 Bids for New York City Casino Licenses

  Topic 3:
    [0.76] [Well] Dementia Comes in Many Forms. Alzheimer’s Is Just One.
    [0.91] [Business] Working From Home Is Harming Young Employees. They’re Starting to See That.

  Topic 4:
    [0.45] [Food] These 7 Cookies Will Be the Life of Every Party
    [0.44] [Travel] Why Is the ‘City of Baths’ Running Out of Bathhouses?

  Topic 5:
    [0.29] [New York] The Transgender Cancer Patient a